<table>
  <tr>
   <td width="220" align="center" style="background:#222; padding:12px;">
    <img src="https://anosys.ai/assets/img/3.png" width="200">
</td>
    <td valign="middle">
      <h1 style="margin:0;">OpenAI Chat - Observability Example</h1>
      <p style="margin:8px 0 0 0;">
        Interactive notebook demonstrating chat applications built with the Anosys SDK and OpenAI.
      </p>
    </td>
  </tr>
</table>


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/Anosys-AI/anosys-sdk/blob/main/examples/anosys_openai_chat_poc.ipynb)

This notebook demonstrates how to use the standard **OpenAI Python Library** in "Chat Mode" and gain deep visibility into its behavior using **AnoSys Observability**.



### What you will learn:
1. How to initialize the AnoSys OpenAI Logger for direct API calls.
2. How to use the standard `chat.completions.create` method.
3. How to capture automatic traces (including tool calls) without the "Agentic" abstraction layer.
4. How to handle streaming responses with observability.

## Step 1: Installation

First, we need to install the necessary libraries. We'll use the standard `openai` library and the `anosys-sdk-openai` for observability.

Visit our library for the latest updates and features:
[anosys-sdk-openai on PyPI](https://pypi.org/project/anosys-sdk-openai)

In [ ]:
!pip install -q anosys-sdk-openai openai

## Step 2: Configuration

To run this PoC, you'll need two API keys:
1. **OpenAI API Key**: To power the completions.
2. **AnoSys API Key**: To send traces to your observability dashboard.

You can get your AnoSys API key from the [AnoSys Console](https://console.anosys.ai) > Data Ingestion > Integration Options.

In [2]:
import os
from google.colab import userdata #Use colab secrets to store your keys

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

if "ANOSYS_API_KEY" not in os.environ:
    os.environ["ANOSYS_API_KEY"] = userdata.get('ANOSYS_API_KEY')

## Step 3: Initialize AnoSys Observability

The `AnosysOpenAILogger` automatically instruments the OpenAI client. Once initialized, all subsequent API calls are captured automatically.

In [ ]:
from anosys_sdk_openai import AnosysOpenAILogger
from openai import OpenAI

# Initialize AnoSys (do this once at startup)
AnosysOpenAILogger()

client = OpenAI()

print("✅ AnoSys OpenAI Observability initialized!")

## Step 4: Simple Chat Completion

Let's run a standard chat completion. Notice we are using the official OpenAI client directly.

In [ ]:
response = client.chat.completions.create(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Why is AnoSys a strong choice for AI observability?"}
    ]
)

print("--- LLM Response ---")
print(response.choices[0].message.content)

print("\n✅ Trace captured! Check your AnoSys dashboard.")

## Step 5: Streaming Response

AnoSys also automatically captures streaming responses.

In [ ]:
print("--- Streaming Response ---")
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Write a short poem about space."}],
    stream=True
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="")

print("\n\n✅ Streaming trace captured!")

## 🛠️ Step 6: Tool Usage (Function Calling)

Even without the Agentic framework, you can use OpenAI tools. AnoSys will capture the tool call requests and the LLM's intent.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"}
                },
                "required": ["location"]
            }
        }
    }
]

print("--- Requesting Tool Call ---")
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What's the weather like in New York?"}],
    tools=tools,
    tool_choice="auto"
)

tool_call = response.choices[0].message.tool_calls[0]
print(f"LLM requested tool: {tool_call.function.name}")
print(f"Arguments: {tool_call.function.arguments}")

print("\n✅ Tool call trace captured!")

## Step 7: Explore Your Traces in AnoSys

Head over to your **AnoSys Dashboard** to see the results.

### What to look for:
1. **Generation Metadata**: View tokens used, model information, and request parameters.
2. **Chat History**: See the exact messages sent and received.
3. **Streaming Timeline**: Visualize the duration of the stream.
4. **Tool Intent**: See the specific tool and arguments the LLM decided to use.